# General Linear Regression and Statistical Inference Lab

## Introduction

You are a junior data scientist working for a real estate investment firm that wants to build a **predictive model** for California housing prices. Your team needs to understand which factors, such as median income, average number of rooms, and average occupancy, most strongly influence **Median House Value**.

Additionally, your manager is interested in whether adding nonlinear terms (such as squared income) can improve the model's accuracy. This analysis will help the company make data-driven investment decisions by identifying key predictors of home prices and refining pricing strategies for different market segments.

In real estate, multiple factors interact to determine property prices. A simple linear model may not always capture complex relationships, so testing for statistical significance and exploring nonlinear effects is crucial. This lab will help you:
- Identify which factors significantly impact house prices.
- Assess the uncertainty of these estimates using confidence intervals.
- Compare the effectiveness of a linear vs. a quadratic model using adjusted $R^2$.

By completing this lab, you will gain experience in **multivariate regression modeling, statistical inference, and model evaluation**, skills that are essential in predictive analytics for business decision-making.

### Step 0

Load the appropriate libraries and bring in the data. Note that we have to run a script to get the [California Housing dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_california_housing.html) to match the version in `scikit-learn`. We cannot pull it directly from `scikit-learn` since CodeGrade cannot access the internet.

In [1]:
# CodeGrade step0

from sklearn.datasets import fetch_california_housing
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from scipy.stats import pearsonr
import os
import tarfile
import joblib # Import joblib directly
from sklearn.datasets._base import _pkl_filepath, get_data_home
import statsmodels.api as sm
import statsmodels.formula.api as smf
import seaborn as sns

# Process to get the data to match the version in scikit-learn:
archive_path = "cal_housing.tgz" # Change the path if it's not in the current directory.
data_home = get_data_home(data_home = None) # Change data_home if you are not using ~/scikit_learn_data.
if not os.path.exists(data_home):
    os.makedirs(data_home)
filepath = _pkl_filepath(data_home, "cal_housing.pkz")

with tarfile.open(mode = "r:gz", name = archive_path) as f:
    cal_housing = np.loadtxt(
                            f.extractfile("CaliforniaHousing/cal_housing.data"),
                            delimiter = ",")
    # Columns are not in the same order compared to the previous URL resource
    # on lib.stat.cmu.edu:
    columns_index = [8, 7, 2, 3, 4, 5, 6, 1, 0]
    cal_housing = cal_housing[:, columns_index]

    joblib.dump(cal_housing, filepath, compress = 6) # Now using the directly imported joblib.

# Load the dataset:
california = fetch_california_housing(as_frame = True)
data = california.data
data["MedianHouseValue"] = california.target

Print the basic information of the data using `.info()` and `.describe()`.

In [2]:
# Display basic information:
print(data.info())
print(data.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   MedInc            20640 non-null  float64
 1   HouseAge          20640 non-null  float64
 2   AveRooms          20640 non-null  float64
 3   AveBedrms         20640 non-null  float64
 4   Population        20640 non-null  float64
 5   AveOccup          20640 non-null  float64
 6   Latitude          20640 non-null  float64
 7   Longitude         20640 non-null  float64
 8   MedianHouseValue  20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB
None
             MedInc      HouseAge      AveRooms     AveBedrms    Population  \
count  20640.000000  20640.000000  20640.000000  20640.000000  20640.000000   
mean       3.870671     28.639486      5.429000      1.096675   1425.476744   
std        1.899822     12.585558      2.474173      0.473911   1132.462122   
min        0.4

### Step 1

- Let the `X` variable be `MedInc`, `AveRooms`, and `AveOccup` and add the constant term for the intercept.
- Let `y` be `MedianHouseValue`.
- Fit the regression model, calling it `mlr_model`.

Return the $r^2$ value of the model rounded to four decimal places.

In [3]:
# CodeGrade step1

X = data[["MedInc", "AveRooms", "AveOccup"]]
y = data[["MedianHouseValue"]]

# Add a constant for the y-intercept term, so that the model doesn't force the
# line through the origin:
X_const = sm.add_constant(X)

print(X_const.shape, y.shape)

# Create the multiple linear regression model using the Ordinary Least Squares
# (OLS) method:
mlr_model = smf.ols(formula = "MedianHouseValue ~ MedInc + AveRooms + " \
                    "AveOccup", data = data).fit()

mlr_r_squared = round(mlr_model.rsquared, 4)
mlr_r_squared

(20640, 4) (20640, 1)


0.4808

Print the multiple linear regression model summary.

In [4]:
print(mlr_model.summary())

                            OLS Regression Results                            
Dep. Variable:       MedianHouseValue   R-squared:                       0.481
Model:                            OLS   Adj. R-squared:                  0.481
Method:                 Least Squares   F-statistic:                     6370.
Date:                Wed, 23 Sep 2026   Prob (F-statistic):               0.00
Time:                        21:29:12   Log-Likelihood:                -25477.
No. Observations:               20640   AIC:                         5.096e+04
Df Residuals:                   20636   BIC:                         5.099e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.6069      0.016     37.444      0.0

### Step 2

- Let `p_values` be the model's p-values.

Return the four p-values using `.iloc[]` from the first value to the fourth, in order, and separated by commas. Make sure to round each to five decimal places.

In [5]:
# CodeGrade step2

p_values = mlr_model.pvalues

round(p_values.iloc[0], 5), round(p_values.iloc[1], 5), \
round(p_values.iloc[2], 5), round(p_values.iloc[3], 5)

(0.0, 0.0, 0.0, 0.0)

### Step 3

- Identify the significant predictors (strictly less than $\alpha = 0.05$), calling this `significant_predictors`.

Return the shape of `significant_predictors`.

In [6]:
# CodeGrade step3

# I am skipping the p-value for the intercept because the instructions say
# "identify the significant PREDICTORS."
significant_predictors = p_values.iloc[1:] < 0.05

significant_predictors.shape

(3,)

### Step 4

- Find the confidence intervals of the model (at a 95% level of confidence), calling this `conf_intervals`.

Using `.iloc[,]` and rounding to two decimal places, return the four confidence intervals in the following order (separated by commas):

`First row and first column, first row and second column, second row and first column, second row and second column`

In [7]:
# CodeGrade step4

conf_intervals = mlr_model.conf_int(alpha = 0.05)

# Return just the start and end points of the first two confidence intervals,
# as the lab instructions say:
round(conf_intervals.iloc[0, 0], 2), round(conf_intervals.iloc[0, 1], 2), \
round(conf_intervals.iloc[1, 0], 2), round(conf_intervals.iloc[1, 1], 2)

(0.58, 0.64, 0.43, 0.44)

Now to see how the intervals look "nicely," return `conf_intervals`.

In [8]:
# This will show all the confidence intervals completely:
conf_intervals

,0,1
Intercept,0.575162,0.638703
MedInc,0.428363,0.441003
AveRooms,-0.043178,-0.033474
AveOccup,-0.005266,-0.003081


### Step 5

- Add a quadratic term to the data called `MedInc_squared`, which is the square of `MedInc`, and call the new model `quad_model`. Make sure to include the variables `MedInc`, `AveRooms`, and `AveOccup` as well.

Return the $r^2$ value of the quadratic model rounded to four decimal places.

In [9]:
# CodeGrade step5

# Add a quadratic term to the data that is the square of `MedInc`:
data[["MedInc_squared"]] = data[["MedInc"]]**2

# Create the quadratic regression model using the Ordinary Least Squares (OLS)
# method:
quad_model = smf.ols(formula = "MedianHouseValue ~ MedInc_squared + MedInc + " \
                     "AveRooms + AveOccup", data = data).fit()

quad_r_squared = round(quad_model.rsquared, 4)
quad_r_squared

0.4858

Print the quadratic regression model summary.

In [10]:
print(quad_model.summary())

                            OLS Regression Results                            
Dep. Variable:       MedianHouseValue   R-squared:                       0.486
Model:                            OLS   Adj. R-squared:                  0.486
Method:                 Least Squares   F-statistic:                     4874.
Date:                Wed, 23 Sep 2026   Prob (F-statistic):               0.00
Time:                        21:29:12   Log-Likelihood:                -25378.
No. Observations:               20640   AIC:                         5.077e+04
Df Residuals:                   20635   BIC:                         5.081e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept          0.3551      0.024     14.

### Step 6

- Find the adjusted $r^2$ values for both models and call them `adjusted_r2_base` and `adjusted_r2_quad`, respectively.

Return these two adjusted $r^2$ values rounded to four decimal places, separated by a comma.

In [11]:
# CodeGrade step6

adjusted_r2_base = mlr_model.rsquared_adj
adjusted_r2_quad = quad_model.rsquared_adj

round(adjusted_r2_base, 4), round(adjusted_r2_quad, 4)

(0.4807, 0.4857)

Print both of these adjusted $r^2$ values.

In [12]:
print(f"Adjusted R-squared for Base Multiple Linear Regression Model: {adjusted_r2_base:.4f}")
print(f"Adjusted R-squared for Quadratic Regression Model: {adjusted_r2_quad:.4f}")

Adjusted R-squared for Base Multiple Linear Regression Model: 0.4807
Adjusted R-squared for Quadratic Regression Model: 0.4857
